In [1]:

import os
os.environ["VLLM_CONFIGURE_LOGGING"] = "0"
import logging
logging.basicConfig(format='%(message)s', level=logging.FATAL)

import json
import importlib
import pprint

import random
import numpy as np

import torch
from vllm import LLM, SamplingParams, PoolingParams

from sal.config import Config

from core import mcts_embeds_search_v03_02_03
# from core import mcts_search_extra_v31
# from core import mcts_search_extra_v61
# from core import mcts_search_extra_v71

from core.reward_models import RLHFFlow
from utils.load_data import load_data_prm800k_hf, load_data_prm800k

from sal.utils.score import aggregate_scores


In [2]:
if torch.cuda.is_available():
    GPUS = os.environ.get('CUDA_VISIBLE_DEVICES', "0").split(',')
    print(GPUS)
else:
    print("CUDA is not available.")

['0']


In [3]:
# base_dir
base_dir = '/groups/chichengz/tnn/datasets/'

# dataset path
data_dir = base_dir + "/prm800k/math_splits"

# llm and prm path
# llm_dir = base_dir + "/Llama-3.2-1B-Instruct-GGUF/Llama-3.2-1B-Instruct.Q4_K_M.gguf"
# prm_dir = base_dir + "/Llama3.1-8B-PRM-Deepseek-Data-GGUF/Llama3.1-8B-PRM-Deepseek-Data.Q4_K_M.gguf"

llm_dir = base_dir + "/Llama-3.2-1B-Instruct"
prm_dir = base_dir + "/Llama3.1-8B-PRM-Deepseek-Data"
# prm_dir = base_dir + "/Llama3.1-8B-PRM-Deepseek-Data-Modified"

In [4]:
llm_total_gpu = 0.2
llm_gpu_memory_utilization = 0.1

llm_vllm = LLM(
    model=llm_dir, 
    tensor_parallel_size=1, 
    # trust_remote_code=True,
    swap_space=16,
    max_model_len=5000,
    gpu_memory_utilization=llm_gpu_memory_utilization,
    enforce_eager=True,
    distributed_executor_backend=None,
    disable_log_stats=True,
    dtype="float16",
    seed=0,
)
print('#--- memory:', torch.cuda.memory_allocated(0)/(1024**3))



[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
#--- memory: 0.0


In [5]:
llm_vllm_embeds = LLM(
    model=llm_dir, 
    tensor_parallel_size=1, 
    # trust_remote_code=True,
    # task="embed",
    runner="pooling",
    swap_space=16,
    max_model_len=5000,
    gpu_memory_utilization=llm_total_gpu-llm_gpu_memory_utilization,
    enforce_eager=True,
    distributed_executor_backend=None,
    disable_log_stats=True,
    dtype="float16",
    seed=0,
)
print('#--- memory:', torch.cuda.memory_allocated(0)/(1024**3))

[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
[Gloo] Rank 0 is connected to 0 peer ranks. Expected number of connected peer ranks is : 0
#--- memory: 0.0


In [6]:
prm = RLHFFlow(model_path=prm_dir, device_map='cuda:0')
print('#--- memory:', torch.cuda.memory_allocated(0)/(1024**3))

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

#--- memory: 14.95752763748169


In [7]:
from collections import deque

def level_order(root, level_thres):
    res = []
    queue = deque([root]) # start with root

    # level 0: root 
    res.append([root.visit_count()])

    while queue:
        level_size = len(queue)
        level_groups = []

        for _ in range(level_size):
            node = queue.popleft()
            children = node.children
            if children:
                level_groups.append([child.visit_count() for child in children])

                for child in children:
                    queue.append(child)

        if level_groups:
            res.append(level_groups)

    return res
    


In [8]:
def tree_to_dict(node):
    return {
        "tag": node.tag,
        "depth": node.depth,
        "q_value": f"{node.q_value():0.5f}",
        "nvisits": node.visit_count(),
        "is_terminal": node.is_terminal,
        "text": node.state["step"],
        "children": [tree_to_dict(child) for child in node.children]
        
    }

In [9]:
#  load data 
data_by_levels = load_data_prm800k(data_dir)

1: 43
2: 90
3: 105
4: 128
5: 134


In [10]:
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# general params
config = Config()
config.agg_strategy = 'last'

config.n = 4                      # number of budgets to be generated per depth
config.beam_width = 4             # number of nodes left after selection
config.lookahead = 0              # don't use it for now
config.max_depths = 20            # max depths, after reaching max_depth then terminate search 
config.sort_completed = False      
config.filter_duplicates = True   # remove any duplicates in the last list of trajs
config.seed = 0
config.date_string = "Aug 1 2025"

config.num_batches = 4
config.step_budget = config.num_batches*config.max_depths 
config.num_phases = 50

config.lam = 0.01
config.use_ppl = True

config.embeds_normalizing = True
config.embeds_strategy = 'avg'
config.embeds_centering = False
config.embeds_mean_dir = "embeds_mean_bon--level-4--v01_0_0--bs-256"

config.cpuct = 2
config.ds_beta = 1.0
config.ds_alpha = 100.0
config.negative_reward = 0

# config.version = "v51"

In [11]:
level = 4                                   # level of difficulty of questions
num_questions = len(data_by_levels[level])  # level 4 has 128 questions
# num_questions = 1
trial_idx = 2
print(f"num_questions = {num_questions}")
print(f"trial = {trial_idx}")

# get batch of questions ['q1', 'q2', ...]
# batch_of_questions = [data_by_levels[level][q_idx]['problem'] for q_idx in range(num_questions)]
q_idx = 0
batch_of_questions = [data_by_levels[level][q_idx]['problem']]
print(batch_of_questions)

num_questions = 128
trial = 2
['The set of points $(x,y,z)$ that satisfy\n\\[2x = 3y = -z\\]is a line.\n\nThe set of points $(x,y,z)$ that satisfy\n\\[6x = -y = -4z\\]is another line.\n\nFind the angle between these lines, in degrees.']


In [18]:
importlib.reload(mcts_embeds_search_v03_02_03)
# config.num_phases = 50

logging.fatal(f"trial = {trial_idx}")

config.lam = 0.01
config.ds_alpha = 100.0
config.ds_beta = 1.0

np.random.seed(100000+trial_idx)
random.seed(100000+trial_idx)
torch.manual_seed(100000+trial_idx)
torch.cuda.manual_seed(100000+trial_idx)

for _, question in enumerate(batch_of_questions):
    agent = mcts_embeds_search_v03_02_03.MCTS(config=config, question=question)
    agent_completions, c_depths, c_phases, c_step_cnts, step_cnt, p, ndepths_arr, cnt_node_max_depth = \
        mcts_embeds_search_v03_02_03.mcts_search(question, agent, config, llm_vllm, llm_vllm_embeds, prm)


trial = 2

-> p = 0

-> d = 0
cand_text_cleaned
## Step 1: Identify the direction vectors of the lines
The direction vector of the first line is given by the coefficients of x, y, and z in the equation 2x = 3y = -z, which are (2, 3, -1). The direction vector of the second line is given by the coefficients of x, y, and z in the equation 6x = -y = -4z, which are (6, -1, -4).
torch.Size([101, 2048])
cand_text_cleaned
## Step 1: Convert the given equations to slope-intercept form
To find the angle between the lines, we first need to convert the given equations to slope-intercept form, which is $y = mx + b$, where $m$ is the slope and $b$ is the y-intercept.
torch.Size([63, 2048])
cand_text_cleaned
## Step 1: Identify the direction vectors of the lines
First, we identify the direction vectors of the two lines. Let's call the direction vector of the first line $\vec{v_1} = (2, 3, -1)$ and the direction vector of the second line $\vec{v_2} = (6, -1, -4)$.
torch.Size([78, 2048])
cand_text_clea

NameError: name 'stop' is not defined

In [ ]:
stop

In [ ]:
importlib.reload(mcts_search_extra_v61)
config.num_phases = 100

logging.fatal(f"trial = {trial_idx}")

config.lam = 0.01
config.ds_alpha = 100.0
config.ds_beta = 1.0

np.random.seed(100000+trial_idx)
random.seed(100000+trial_idx)
torch.manual_seed(100000+trial_idx)
torch.cuda.manual_seed(100000+trial_idx)

for _, question in enumerate(batch_of_questions):
    agent = mcts_search_extra_v61.MCTS(config=config, question=question)
    agent_completions, c_depths, c_phases, c_step_cnts, step_cnt, p, ndepths_arr, cnt_node_max_depth = \
        mcts_search_extra_v61.mcts_search(question, agent, config, llm_vllm, llm_vllm_embeds, prm)


In [ ]:
version = "v61"
tree_dict = tree_to_dict(agent.root)
# pprint.pprint(tree_dict, indent=2)
with open(f"tree_level-{level}--qidx-{q_idx}--{version}--d-{config.max_depths}--lam-{config.lam}--alpha-{config.ds_alpha}--trial-{trial_idx}.json", 'w', encoding = 'utf-8') as fout:
    json.dump(tree_dict, fout)
    fout.write('\n') 

In [ ]:
print(f"step_cnt = {step_cnt}")
print(f"ncomps = {len(agent_completions)}")
c_nvisits = level_order(agent.root, level_thres=3)
print(f"nvisits = {c_nvisits}")
print(f"total_nphases = {p}")
c_nphases_mean = np.mean(c_phases)
c_nphases_std = np.std(c_phases, ddof=1)/np.sqrt(len(c_phases))
print(f"c_nphases = {c_phases} | {c_nphases_mean:0.1f} (\u00B1{c_nphases_std:0.1f})")
c_ndepths_mean = np.mean(c_depths)
c_ndepths_std = np.std(c_depths, ddof=1)/np.sqrt(len(c_depths))
print(f"c_ndepths = {c_depths} | {c_ndepths_mean:0.1f} (\u00B1{c_ndepths_std:0.1f})")


In [ ]:
for com_idx, comp in enumerate(agent_completions):
    print(f"\n-> com_idx = {com_idx}")
    print(comp)

In [ ]:
importlib.reload(mcts_search_extra_v31)

config.num_phases = 50

logging.fatal(f"trial = {trial_idx}")

np.random.seed(100000+trial_idx)
random.seed(100000+trial_idx)
torch.manual_seed(100000+trial_idx)
torch.cuda.manual_seed(100000+trial_idx)

for _, question in enumerate(batch_of_questions):
    agent = mcts_search_extra_v31.MCTS(config=config, question=question)
    agent_completions, c_depths, c_phases, c_step_cnts, step_cnt, p, ndepths_arr, cnt_node_max_depth = \
        mcts_search_extra_v31.mcts_search(question, agent, config, llm_vllm, prm)



In [ ]:
importlib.reload(mcts_search_extra_v21)

config.num_phases = 50

logging.fatal(f"trial = {trial_idx}")

np.random.seed(100000+trial_idx)
random.seed(100000+trial_idx)
torch.manual_seed(100000+trial_idx)
torch.cuda.manual_seed(100000+trial_idx)

for _, question in enumerate(batch_of_questions):
    agent = mcts_search_extra_v21.MCTS(config=config, question=question)
    agent_completions, c_depths, c_phases, c_step_cnts, step_cnt, p, ndepths_arr, cnt_node_max_depth = \
        mcts_search_extra_v21.mcts_search(question, agent, config, llm_vllm, prm)

In [ ]:
version = "v21"
tree_dict = tree_to_dict(agent.root)
# pprint.pprint(tree_dict, indent=2)
with open(f"tree_level-{level}--qidx-{q_idx}--{version}--d-{config.max_depths}--trial-{trial_idx}.json", 'w', encoding = 'utf-8') as fout:
    json.dump(tree_dict, fout)
    fout.write('\n') 

In [ ]:
print(f"step_cnt = {step_cnt}")
print(f"ncomps = {len(agent_completions)}")
c_nvisits = level_order(agent.root, level_thres=3)
print(f"nvisits = {c_nvisits}")
print(f"total_nphases = {p}")
c_nphases_mean = np.mean(c_phases)
c_nphases_std = np.std(c_phases, ddof=1)/np.sqrt(len(c_phases))
print(f"c_nphases = {c_phases} | {c_nphases_mean:0.1f} (\u00B1{c_nphases_std:0.1f})")
c_ndepths_mean = np.mean(c_depths)
c_ndepths_std = np.std(c_depths, ddof=1)/np.sqrt(len(c_depths))
print(f"c_ndepths = {c_depths} | {c_ndepths_mean:0.1f} (\u00B1{c_ndepths_std:0.1f})")

In [ ]:
for com_idx, comp in enumerate(agent_completions):
    print(f"\n-> com_idx = {com_idx}")
    print(comp)
